# 03 — Live Open-Meteo sanity check

**Purpose:** verify that the one active provider returns current Karak air-quality and weather observations with the expected timezone and fields. This is not a cross-provider comparison.


In [1]:
from pathlib import Path
from datetime import date, timedelta
import sys
import requests
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = (PROJECT_ROOT / '..').resolve()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from config import LATITUDE, LONGITUDE, CITY_NAME, TIMEZONE, OPEN_METEO_AIR_QUALITY_URL, OPEN_METEO_WEATHER_FORECAST_URL

def get(url, params):
    response = requests.get(url, params=params, timeout=60)
    response.raise_for_status()
    return response.json()

today = date.today().isoformat()
air = get(OPEN_METEO_AIR_QUALITY_URL, {'latitude': LATITUDE, 'longitude': LONGITUDE, 'hourly': 'pm2_5,pm10,ozone,us_aqi', 'forecast_days': 2, 'timezone': TIMEZONE})
weather = get(OPEN_METEO_WEATHER_FORECAST_URL, {'latitude': LATITUDE, 'longitude': LONGITUDE, 'hourly': 'temperature_2m,relative_humidity_2m,wind_speed_10m,precipitation', 'forecast_days': 2, 'timezone': TIMEZONE})
print('Target:', CITY_NAME, LATITUDE, LONGITUDE)
print('API timezone:', air.get('timezone'), '| UTC offset seconds:', air.get('utc_offset_seconds'))
print('Air response hours:', len(air['hourly']['time']))
print('Weather response hours:', len(weather['hourly']['time']))
air_frame = pd.DataFrame(air['hourly'])
weather_frame = pd.DataFrame(weather['hourly'])
print('Air timestamps:', air_frame['time'].min(), '->', air_frame['time'].max())
print('Weather timestamps:', weather_frame['time'].min(), '->', weather_frame['time'].max())
print('Air missing cells:', int(air_frame.isna().sum().sum()))
print('Weather missing cells:', int(weather_frame.isna().sum().sum()))
print('Live Open-Meteo verdict:', 'PASS' if air.get('timezone') == TIMEZONE and weather.get('timezone') == TIMEZONE and not air_frame.isna().any().any() and not weather_frame.isna().any().any() else 'REVIEW')


Target: Karak 33.1383653 71.1909136
API timezone: Asia/Karachi | UTC offset seconds: 18000
Air response hours: 48
Weather response hours: 48
Air timestamps: 2026-07-31T00:00 -> 2026-08-01T23:00
Weather timestamps: 2026-07-31T00:00 -> 2026-08-01T23:00
Air missing cells: 0
Weather missing cells: 0
Live Open-Meteo verdict: PASS


### Finding from the live request

The cell above records the actual endpoint response, local timezone, row counts, date ranges, missing-cell counts, and a pass/review verdict. Because both requests are Open-Meteo, the check verifies operational consistency without introducing a secondary data source.
